# **Using *MiniLMv2-L12-MNLI/XNLI* for New Fields Creation**

# All Imports Here

In [39]:
import os
glb_pth = 'd:\\GitHub\\SaaS_Product_Analysis'
os.chdir(glb_pth)

import re
import pandas as pd
import json
import torch
from pathlib import Path
from transformers import pipeline

# Importing Datasets

In [40]:
data_pth = 'data\\cleaned'

zoom_df = pd.read_csv(f'{glb_pth}\\{data_pth}\\zoom_reviews_clean.csv')
meet_df = pd.read_csv(f'{glb_pth}\\{data_pth}\\meet_reviews_clean.csv')
teams_df = pd.read_csv(f'{glb_pth}\\{data_pth}\\teams_reviews_clean.csv')
webex_df = pd.read_csv(f'{glb_pth}\\{data_pth}\\webex_reviews_clean.csv')

# Implementing Model

## *Testing MiniLMv2-L12-MNLI/XNLI*

In [41]:
pipe = pipeline(
    'zero-shot-classification',
    model='MoritzLaurer/multilingual-MiniLMv2-L12-mnli-xnli'
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7237.89it/s]


In [42]:
sequence_to_classify = (
    "🤬🤬🤬"
)

candidate_labels = ['positive', 'negative', 'neutral']

output = pipe(sequence_to_classify, candidate_labels)

max_prob = max(output['scores'])
max_idx = output['scores'].index(max_prob)

print(f'Prediction: {output['labels'][max_idx].capitalize()}\nProb: {round(max_prob, 2)*100.0}%')
print(output)

Prediction: Positive
Prob: 62.0%
{'sequence': '🤬🤬🤬', 'labels': ['positive', 'negative', 'neutral'], 'scores': [0.6238628029823303, 0.23767931759357452, 0.13845789432525635]}


## *Using on datasets*

In [52]:
def sentiment_prediction(body):
    candidate_labels = [
        'positive feedback or satisfaction', 
        'dissatisfaction, complaint, failure, or product problem', 
        'neutral feedback without praise or complaint'
    ]
    probs = []
    labels = []
    for i, texts in enumerate(body):
        output = pipe(
            texts, 
            candidate_labels,
            hypothesis_template="The customer is expressing {} about their experience with the product.",
            multi_label=False
        )
        max_prob = max(output['scores'])
        max_label = output['labels'][
            output['scores'].index(max_prob)
        ]
        
        probs.append(max_prob)
        if max_prob < 0.55:
            labels.append('Uncertain')
        elif max_label == 'positive feedback or satisfaction':
            labels.append('positive')
        elif max_label == 'dissatisfaction, complaint, failure, or product problem':
            labels.append('negative')
        else:
            labels.append('neutral')

        if i % 100 == 0:
            print(f'Processed {i} reviews')
    return (probs, labels)

In [53]:
zoom_df['sentiment_score'], zoom_df['sentiment_label'] = (
    sentiment_prediction(zoom_df['body'])
)

Processed 0 reviews
Processed 100 reviews
Processed 200 reviews
Processed 300 reviews
Processed 400 reviews
Processed 500 reviews
Processed 600 reviews
Processed 700 reviews
Processed 800 reviews
Processed 900 reviews
Processed 1000 reviews
Processed 1100 reviews
Processed 1200 reviews
Processed 1300 reviews
Processed 1400 reviews
Processed 1500 reviews
Processed 1600 reviews
Processed 1700 reviews
Processed 1800 reviews
Processed 1900 reviews
Processed 2000 reviews
Processed 2100 reviews
Processed 2200 reviews
Processed 2300 reviews
Processed 2400 reviews
Processed 2500 reviews
Processed 2600 reviews
Processed 2700 reviews
Processed 2800 reviews
Processed 2900 reviews
Processed 3000 reviews
Processed 3100 reviews
Processed 3200 reviews
Processed 3300 reviews
Processed 3400 reviews
Processed 3500 reviews
Processed 3600 reviews
Processed 3700 reviews
Processed 3800 reviews
Processed 3900 reviews
Processed 4000 reviews
Processed 4100 reviews
Processed 4200 reviews
Processed 4300 reviews


In [54]:
zoom_df.head(20)


,Unnamed: 0,reviewId,rating,body,appVersion,timestamp,sentiment_score,sentiment_label
0,0,0dcdf6d4-dfa0-4401-b3d4-d9dbc27c825f,1,👎👎👎👎👎,NaN,2026-08-26 05:50:54,0.471210,Uncertain
1,1,d70e1f1c-83fc-495b-8035-3eb460431f82,1,Can't login after resignation,NaN,2026-08-26 05:50:12,0.401677,Uncertain
2,2,ad750e18-d0d4-40cc-8797-26e77acdbfe4,1,Worst app,NaN,2026-08-26 05:49:41,0.781650,negative
3,3,359d855e-2a8b-4313-8120-91dff1c872ef,1,Can't login,NaN,2026-08-26 05:49:09,0.709888,negative
4,4,dae23cb0-c77e-4f78-89ea-72b01e4c7799,1,Can't login after resignation,NaN,2026-08-26 05:48:36,0.401677,Uncertain
5,5,a6aea44a-1e7a-40a0-900c-f29ea807b1aa,1,Can't login after resignation,NaN,2026-08-26 05:48:10,0.401677,Uncertain
6,6,c819e071-cbc7-4bf2-88c6-3cb5af6e970a,1,Tôi đã nhập đúng passcode nhưng ứng dụng lại t...,7.1.6.41876,2026-08-26 05:46:55,0.970556,negative
7,7,ef8e0a4d-9ec1-4edc-9868-5040eed5be7d,5,very good,NaN,2026-08-26 05:15:05,0.831097,positive
8,8,29d469f8-6259-4055-859b-270e6ac0aaf6,1,why i cant get in any zoom meetings? please fi...,7.1.6.41876,2026-08-26 04:56:37,0.705390,negative
9,9,d8d05f54-6fae-4a90-8d58-9d2d53163965,5,👍👍👌👌🇮🇳🇮🇳🎉🎉👌👌,7.1.6.41876,2026-08-26 04:32:55,0.590357,positive


In [55]:
teams_df['sentiment_score'], teams_df['sentiment_label'] = (
    sentiment_prediction(teams_df['body'])
)

Processed 0 reviews
Processed 100 reviews
Processed 200 reviews
Processed 300 reviews
Processed 400 reviews
Processed 500 reviews
Processed 600 reviews
Processed 700 reviews
Processed 800 reviews
Processed 900 reviews
Processed 1000 reviews
Processed 1100 reviews
Processed 1200 reviews
Processed 1300 reviews
Processed 1400 reviews
Processed 1500 reviews
Processed 1600 reviews
Processed 1700 reviews
Processed 1800 reviews
Processed 1900 reviews
Processed 2000 reviews
Processed 2100 reviews
Processed 2200 reviews
Processed 2300 reviews
Processed 2400 reviews
Processed 2500 reviews
Processed 2600 reviews
Processed 2700 reviews
Processed 2800 reviews
Processed 2900 reviews
Processed 3000 reviews
Processed 3100 reviews
Processed 3200 reviews
Processed 3300 reviews
Processed 3400 reviews
Processed 3500 reviews
Processed 3600 reviews
Processed 3700 reviews
Processed 3800 reviews
Processed 3900 reviews
Processed 4000 reviews
Processed 4100 reviews
Processed 4200 reviews
Processed 4300 reviews


In [56]:
teams_df.head()

,Unnamed: 0,reviewId,rating,body,appVersion,timestamp,sentiment_score,sentiment_label
0,0,c7424e1f-784c-4bea-a863-93cbf4d6a0d9,1,"It literally won't load, it's just useless on ...",1416/1.0.0.2026142702,2026-08-26 07:07:02,0.435989,Uncertain
1,1,b0c158ef-e997-4489-a696-64874318d661,5,it's de best app for online meetings 🥰,1416/1.0.0.2026142702,2026-08-26 06:22:02,0.866845,positive
2,2,dce12548-3288-4fcb-94f2-47361c5de3d4,5,Great app,1416/1.0.0.2026142702,2026-08-26 06:13:26,0.795333,positive
3,3,6bd53442-17bc-4d3d-ae16-2f00feb338eb,1,i am facing glitch during calls suddenly sound...,1416/1.0.0.2026142702,2026-08-26 05:54:12,0.760752,negative
4,4,f0f2fd5d-b67a-486e-8b5e-7de5fd218e14,1,totally waste my college account never login t...,1416/1.0.0.2026142702,2026-08-26 05:53:07,0.733569,negative


In [57]:
meet_df['sentiment_score'], meet_df['sentiment_label'] = (
    sentiment_prediction(meet_df['body'])
)

Processed 0 reviews
Processed 100 reviews
Processed 200 reviews
Processed 300 reviews
Processed 400 reviews
Processed 500 reviews
Processed 600 reviews
Processed 700 reviews
Processed 800 reviews
Processed 900 reviews
Processed 1000 reviews
Processed 1100 reviews
Processed 1200 reviews
Processed 1300 reviews
Processed 1400 reviews
Processed 1500 reviews
Processed 1600 reviews
Processed 1700 reviews
Processed 1800 reviews
Processed 1900 reviews
Processed 2000 reviews
Processed 2100 reviews
Processed 2200 reviews
Processed 2300 reviews
Processed 2400 reviews
Processed 2500 reviews
Processed 2600 reviews
Processed 2700 reviews
Processed 2800 reviews
Processed 2900 reviews
Processed 3000 reviews
Processed 3100 reviews
Processed 3200 reviews
Processed 3300 reviews
Processed 3400 reviews
Processed 3500 reviews
Processed 3600 reviews
Processed 3700 reviews
Processed 3800 reviews
Processed 3900 reviews
Processed 4000 reviews
Processed 4100 reviews
Processed 4200 reviews
Processed 4300 reviews


In [58]:
meet_df.head()

,Unnamed: 0,reviewId,rating,body,appVersion,timestamp,sentiment_score,sentiment_label
0,0,425f3803-d4e3-4cae-abb5-6b6aff5a0ffb,5,Malinaw at matagal maputol ang tawag.,371.0.958218800.duo.android_20260803.01_p0,2026-08-26 07:20:13,0.505840,Uncertain
1,1,79aab19f-bddd-4ceb-a88f-0812e1e882c4,5,nice,371.0.963601321.duo.android_20260803.01_p4,2026-08-26 07:15:24,0.765256,positive
2,2,81533c07-6961-40da-aaf4-57874055c464,5,Masud Aalam,372.0.968082697.duo.android_20260810.02_p4,2026-08-26 05:36:24,0.523674,Uncertain
3,3,a889eea4-571f-4e34-a49d-babfcb3b7aee,5,It works and I think it works very well.,371.0.958218800.duo.android_20260803.01_p0,2026-08-26 05:06:07,0.879606,positive
4,4,38154815-a509-4214-a9bd-20e49d424b69,1,LISTEN TO THE DHAMMA TO KNOW THE TRUTH သစၥာသိဖ...,371.0.963601321.duo.android_20260803.01_p4,2026-08-26 04:34:37,0.779508,negative


In [59]:
webex_df.dropna(inplace=True)
webex_df['sentiment_score'], webex_df['sentiment_label'] = (
    sentiment_prediction(webex_df['body'])
)

Processed 0 reviews
Processed 100 reviews
Processed 200 reviews
Processed 300 reviews
Processed 400 reviews
Processed 500 reviews
Processed 600 reviews
Processed 700 reviews
Processed 800 reviews
Processed 900 reviews
Processed 1000 reviews
Processed 1100 reviews
Processed 1200 reviews
Processed 1300 reviews
Processed 1400 reviews
Processed 1500 reviews
Processed 1600 reviews
Processed 1700 reviews
Processed 1800 reviews
Processed 1900 reviews
Processed 2000 reviews
Processed 2100 reviews
Processed 2200 reviews
Processed 2300 reviews
Processed 2400 reviews
Processed 2500 reviews
Processed 2600 reviews
Processed 2700 reviews
Processed 2800 reviews
Processed 2900 reviews
Processed 3000 reviews
Processed 3100 reviews
Processed 3200 reviews
Processed 3300 reviews
Processed 3400 reviews
Processed 3500 reviews
Processed 3600 reviews
Processed 3700 reviews
Processed 3800 reviews
Processed 3900 reviews
Processed 4000 reviews
Processed 4100 reviews
Processed 4200 reviews
Processed 4300 reviews


In [60]:
webex_df.head()

,Unnamed: 0,reviewId,rating,body,appVersion,timestamp,sentiment_score,sentiment_label
0,0,5357375b-b0f3-4911-8d9f-e578967960c1,4,less used than zooms,45.3.0,2026-08-25 18:13:04,0.531206,Uncertain
1,1,df121414-03e3-4dcf-91d3-9dcb7c265822,5,Better,45.3.0,2026-08-25 10:49:21,0.788017,positive
4,4,49290776-d039-45f6-a923-121ce46c2a12,1,Not useful for the host. You can't schedule or...,45.3.0,2026-08-23 01:19:47,0.584916,negative
7,7,3c60b219-a3aa-462c-8411-7658d8cef51a,5,great...,45.3.0,2026-08-20 13:30:39,0.661484,positive
8,8,e23d8dce-ec99-4741-8d5d-3ec6cda50d3b,4,nice app,45.3.0,2026-08-18 09:11:58,0.706628,positive
